In [ ]:
# Safe Jupyter startup on Linux only for spawning figures and windows 
import os, sys

if sys.platform.startswith("linux"):
    os.environ.setdefault("MPLBACKEND", "Agg")        # avoids Qt/OpenGL issues
    os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
    #os.environ.setdefault("OMP_NUM_THREADS", "1")     # tames OpenMP storms
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
    #os.environ.setdefault("CUDA_VISIBLE_DEVICES", "") # comment out if you WANT GPU
else:
    # macOS/Windows: leave defaults; optionally keep OMP cap
    #os.environ.setdefault("OMP_NUM_THREADS", "1")
    pass


## Saccade Quantification (loads processed outputs from 1_2)

In [ ]:
import numpy as np
from pathlib import Path
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gc
import io
import json

from scipy.stats import linregress
from scipy.signal import butter, filtfilt
from scipy.signal import correlate
from scipy.stats import pearsonr

from harp_resources import process
import aeon.io.api as api
from sleap import saccade_processing as sp
from sleap.saccade_processing import analyze_eye_video_saccades
from sleap.visualization import plot_all_saccades_overlay, plot_saccade_amplitude_qc
from sleap.annotation_gui import launch_annotation_gui
from sleap.ml_feature_extraction import extract_experiment_id
from sleap.annotation_storage import load_annotations, print_annotation_stats
from sleap.visualization import visualize_ml_features
from sleap.ml_feature_extraction import extract_ml_features

# Reload modules to pick up latest changes (useful after code updates)
# Set force_reload_modules = True to always reload, or False to use cached
# versions
# Set to False for faster execution when modules haven't changed
force_reload_modules = False
if force_reload_modules:
    import importlib
    import sleap.load_and_process
    import sleap.processing_functions
    import sleap.saccade_processing
    import sleap.visualization
    import sleap.annotation_gui
    import sleap.ml_feature_extraction
    import sleap.annotation_storage

    importlib.reload(sleap.load_and_process)
    importlib.reload(sleap.processing_functions)
    importlib.reload(sleap.saccade_processing)
    importlib.reload(sleap.visualization)
    importlib.reload(sleap.annotation_gui)
    importlib.reload(sleap.ml_feature_extraction)
    importlib.reload(sleap.annotation_storage)
    # Re-import aliases after reload
    sp = sleap.saccade_processing
    from sleap.saccade_processing import analyze_eye_video_saccades
    from sleap.visualization import plot_all_saccades_overlay, plot_saccade_amplitude_qc
    from sleap.annotation_gui import launch_annotation_gui
    from sleap.ml_feature_extraction import extract_experiment_id
    from sleap.annotation_storage import load_annotations, print_annotation_stats

SACCADE_EXPORT_DROP_COLUMNS = {
    "saccade_id",
    "video_key",
    "eye",
    "direction",
    "direction_label",
    "saccade_start_time",
    "saccade_end_time",
    "saccade_peak_time",
    "saccade_start_frame_idx",
    "saccade_peak_frame_idx",
    "saccade_end_frame_idx",
    "saccade_peak_velocity",
    "saccade_amplitude",
    "saccade_displacement",
    "saccade_duration",
    "saccade_start_position",
    "saccade_end_position",
    "saccade_type",
    "bout_id",
    "bout_size",
    "pre_saccade_mean_velocity",
    "pre_saccade_position_drift",
    "post_saccade_position_variance",
    "post_saccade_position_change",
    "classification_confidence",
    "rule_based_class",
    "rule_based_confidence",
    "merge_frame_idx",
}

def get_eye_label(key):
    """Return mapped user-viewable eye label for video key."""
    return VIDEO_LABELS.get(key, key)


def _needs_eye_suffix(column: str) -> bool:
    """Return True if the column should be tagged as eye data during resampling."""
    return any(column.startswith(prefix) for prefix in _SLEAP_EYE_PREFIXES)


def resample_video_dataframe(
    video_df: pd.DataFrame,
    eye_tag: str,
    target_freq_hz: float = None,
    optical_filter_hz: float = None,
) -> pd.DataFrame:
    """Resample a SLEAP video dataframe onto the common time grid."""
    if "Seconds" not in video_df.columns:
        raise ValueError("Video dataframe must contain a 'Seconds' column before resampling.")

    target_freq_hz = target_freq_hz or COMMON_RESAMPLED_RATE
    df_for_resample = video_df.copy()

    # Rename SLEAP-specific columns so the resampler treats them as eye data
    rename_map = {
        col: f"{col}_{eye_tag}"
        for col in df_for_resample.columns
        if col not in {"Seconds", "frame_idx"} and _needs_eye_suffix(col)
    }
    df_for_resample = df_for_resample.rename(columns=rename_map)

    # Convert Seconds to aeon datetime index and drop Seconds before resampling
    df_for_resample.index = pd.to_datetime(df_for_resample["Seconds"].apply(api.aeon))
    df_for_resample = df_for_resample.drop(columns=["Seconds"])

    resampled = process.resample_dataframe(
        df_for_resample,
        target_freq_Hz=target_freq_hz,
        optical_filter_Hz=optical_filter_hz,
    )

    # Restore original column names
    inverse_map = {v: k for k, v in rename_map.items()}
    resampled = resampled.rename(columns=inverse_map)

    # Convert index back to Seconds
    resampled_seconds = (resampled.index - datetime(1904, 1, 1)).total_seconds()
    resampled = resampled.reset_index(drop=True)
    resampled.insert(0, "Seconds", resampled_seconds)

    # Frame indices should remain integers if present
    if "frame_idx" in resampled.columns:
        resampled["frame_idx"] = (
            pd.to_numeric(resampled["frame_idx"], errors="coerce").round().astype("Int64")
        )

    return resampled


def set_aeon_index(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of the dataframe with a DatetimeIndex derived from Seconds."""
    df_indexed = df.copy()
    df_indexed.index = pd.to_datetime(df_indexed["Seconds"].apply(api.aeon))
    df_indexed.index.name = "aeon_time"
    return df_indexed


def append_aeon_time_column(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of the dataframe with an additional aeon_time ISO timestamp column."""
    df_with_time = df.copy()
    if isinstance(df_with_time.index, pd.DatetimeIndex):
        df_with_time["aeon_time"] = df_with_time.index.map(lambda ts: ts.isoformat())
    else:
        df_with_time["aeon_time"] = df_with_time["Seconds"].apply(lambda x: api.aeon(x).isoformat())
    return df_with_time

# symbols to use ✅ ℹ️ ⚠️ ❗

In [ ]:
# Load processed eye-tracking data generated by 1_2_SLEAP_processing.ipynb
##########################################################################

data_path = Path(
    "/Users/rancze/Documents/Data/vestVR/20250409_Cohort3_rotation/Visual_mismatch_day4/B6J2783-2025-04-28T14-57-30"
)

# Saccade analysis controls
video1_eye = "L"  # Options: 'L' or 'R'
debug = True
plot_saccade_detection_QC = True

k1 = 4.5
k2 = 4.5
refractory_period = 0.1
bout_window = 1.5

onset_offset_fraction = 0.2
pre_saccade_window_time = 0.15
post_saccade_window_time = 0.5
baseline_window_start_time = -0.06
baseline_window_end_time = -0.02
smoothing_window_time = 0.08
peak_width_time = 0.005
min_saccade_duration = 0.2

classify_orienting_compensatory = True
pre_saccade_window = 0.3
max_intersaccade_interval_for_classification = 1.0
pre_saccade_velocity_threshold = 50.0
pre_saccade_drift_threshold = 10.0
post_saccade_variance_threshold = 100.0
post_saccade_position_change_threshold_percent = 50.0
use_adaptive_thresholds = True
adaptive_percentile_pre_velocity = 75
adaptive_percentile_pre_drift = 75
adaptive_percentile_post_variance = 25

COMMON_RESAMPLED_RATE = 1000

video2_eye = "R" if video1_eye == "L" else "L"
eye_fullname = {"L": "Left", "R": "Right"}
VIDEO_LABELS = {
    "VideoData1": f"VideoData1 ({video1_eye}: {eye_fullname[video1_eye]})",
    "VideoData2": f"VideoData2 ({video2_eye}: {eye_fullname[video2_eye]})",
}

save_path = data_path.parent / f"{data_path.name}_processedData"
qc_debug_dir = save_path / "QC_and_debug"
qc_debug_dir.mkdir(parents=True, exist_ok=True)

downsampled_output_dir = save_path / "downsampled_data"
metadata_path = downsampled_output_dir / "saccade_input_metadata.json"

metadata = {}
if metadata_path.exists():
    with open(metadata_path, "r") as f:
        metadata = json.load(f)

# Allow metadata to override eye mapping when available.
if metadata.get("video1_eye") in {"L", "R"}:
    video1_eye = metadata["video1_eye"]
    video2_eye = "R" if video1_eye == "L" else "L"
    VIDEO_LABELS = {
        "VideoData1": f"VideoData1 ({video1_eye}: {eye_fullname[video1_eye]})",
        "VideoData2": f"VideoData2 ({video2_eye}: {eye_fullname[video2_eye]})",
    }


def resolve_video_parquet(video_key):
    """Prefer processed parquet when available; fallback to resampled parquet."""
    outputs = metadata.get("outputs", {}).get(video_key, {}) if isinstance(metadata, dict) else {}

    candidates = []
    if isinstance(outputs, dict):
        processed = outputs.get("processed_parquet")
        resampled = outputs.get("resampled_parquet")
        if processed:
            candidates.append(Path(processed))
        if resampled:
            candidates.append(Path(resampled))

    # Backward-compatible default locations
    candidates.extend(
        [
            downsampled_output_dir / f"{video_key}_processed.parquet",
            downsampled_output_dir / f"{video_key}_resampled.parquet",
        ]
    )

    seen = set()
    unique_candidates = []
    for c in candidates:
        key = str(c)
        if key not in seen:
            unique_candidates.append(c)
            seen.add(key)

    for c in unique_candidates:
        if c.exists():
            source_kind = "processed" if c.name.endswith("_processed.parquet") else "resampled"
            return c, source_kind
    return None, None

v1_path, v1_source_kind = resolve_video_parquet("VideoData1")
v2_path, v2_source_kind = resolve_video_parquet("VideoData2")

VideoData1_Has_Sleap = v1_path is not None
VideoData2_Has_Sleap = v2_path is not None

if VideoData1_Has_Sleap:
    VideoData1 = pd.read_parquet(v1_path)
    VideoData1 = VideoData1.reset_index(drop=True)
    FPS_1 = (
        metadata.get("fps", {}).get("VideoData1")
        or (1 / VideoData1["Seconds"].diff().median())
    )
    print(f"✅ Loaded VideoData1 ({v1_source_kind}) from {v1_path}")
    print(f"   FPS_1 = {FPS_1:.3f}")
else:
    print("⚠️ Missing VideoData1 parquet (processed/resampled)")

if VideoData2_Has_Sleap:
    VideoData2 = pd.read_parquet(v2_path)
    VideoData2 = VideoData2.reset_index(drop=True)
    FPS_2 = (
        metadata.get("fps", {}).get("VideoData2")
        or (1 / VideoData2["Seconds"].diff().median())
    )
    print(f"✅ Loaded VideoData2 ({v2_source_kind}) from {v2_path}")
    print(f"   FPS_2 = {FPS_2:.3f}")
else:
    print("⚠️ Missing VideoData2 parquet (processed/resampled)")

if not VideoData1_Has_Sleap and not VideoData2_Has_Sleap:
    raise FileNotFoundError(
        "No eye parquet files found (processed or resampled). Run 1_2_SLEAP_processing.ipynb first."
    )



In [ ]:
saccade_results = {}

# Helper: map detected directions (upward/downward) to NT/TN based on eye assignment
# Left eye: upward→NT, downward→TN; Right eye: upward→TN, downward→NT


def get_direction_map_for_video(video_key):
    eye = video1_eye if video_key == "VideoData1" else video2_eye
    if eye == "L":
        return {"upward": "NT", "downward": "TN"}
    else:
        return {"upward": "TN", "downward": "NT"}


def get_eye_code_for_video(video_key):
    """Return 'L' or 'R' for the specified video."""
    return video1_eye if video_key == "VideoData1" else video2_eye


if VideoData1_Has_Sleap:
    df1 = VideoData1[["Ellipse.Center.X", "Seconds", "frame_idx"]].copy()
    dir_map_v1 = get_direction_map_for_video("VideoData1")
    saccade_results["VideoData1"] = analyze_eye_video_saccades(
        df1,
        FPS_1,
        get_eye_label("VideoData1"),
        k=k1,
        refractory_period=refractory_period,
        onset_offset_fraction=onset_offset_fraction,
        pre_saccade_window_time=pre_saccade_window_time,
        post_saccade_window_time=post_saccade_window_time,
        baseline_window_start_time=baseline_window_start_time,
        baseline_window_end_time=baseline_window_end_time,
        smoothing_window_time=smoothing_window_time,
        peak_width_time=peak_width_time,
        min_saccade_duration=min_saccade_duration,
        upward_label=dir_map_v1["upward"],
        downward_label=dir_map_v1["downward"],
        classify_orienting_compensatory=classify_orienting_compensatory,
        bout_window=bout_window,
        pre_saccade_window=pre_saccade_window,
        max_intersaccade_interval_for_classification=max_intersaccade_interval_for_classification,
        pre_saccade_velocity_threshold=pre_saccade_velocity_threshold,
        pre_saccade_drift_threshold=pre_saccade_drift_threshold,
        post_saccade_variance_threshold=post_saccade_variance_threshold,
        post_saccade_position_change_threshold_percent=post_saccade_position_change_threshold_percent,
        use_adaptive_thresholds=use_adaptive_thresholds,
        adaptive_percentile_pre_velocity=adaptive_percentile_pre_velocity,
        adaptive_percentile_pre_drift=adaptive_percentile_pre_drift,
        adaptive_percentile_post_variance=adaptive_percentile_post_variance,
        debug=debug,
    )


if VideoData2_Has_Sleap:
    df2 = VideoData2[["Ellipse.Center.X", "Seconds", "frame_idx"]].copy()
    dir_map_v2 = get_direction_map_for_video("VideoData2")
    saccade_results["VideoData2"] = analyze_eye_video_saccades(
        df2,
        FPS_2,
        get_eye_label("VideoData2"),
        k=k2,
        refractory_period=refractory_period,
        onset_offset_fraction=onset_offset_fraction,
        pre_saccade_window_time=pre_saccade_window_time,
        post_saccade_window_time=post_saccade_window_time,
        baseline_window_start_time=baseline_window_start_time,
        baseline_window_end_time=baseline_window_end_time,
        smoothing_window_time=smoothing_window_time,
        peak_width_time=peak_width_time,
        min_saccade_duration=min_saccade_duration,
        upward_label=dir_map_v2["upward"],
        downward_label=dir_map_v2["downward"],
        classify_orienting_compensatory=classify_orienting_compensatory,
        bout_window=bout_window,
        pre_saccade_window=pre_saccade_window,
        max_intersaccade_interval_for_classification=max_intersaccade_interval_for_classification,
        pre_saccade_velocity_threshold=pre_saccade_velocity_threshold,
        pre_saccade_drift_threshold=pre_saccade_drift_threshold,
        post_saccade_variance_threshold=post_saccade_variance_threshold,
        post_saccade_position_change_threshold_percent=post_saccade_position_change_threshold_percent,
        use_adaptive_thresholds=use_adaptive_thresholds,
        adaptive_percentile_pre_velocity=adaptive_percentile_pre_velocity,
        adaptive_percentile_pre_drift=adaptive_percentile_pre_drift,
        adaptive_percentile_post_variance=adaptive_percentile_post_variance,
        debug=debug,
    )

In [ ]:
# SAVE SACCADE RESULTS TO FILES
##########################################################################
# Build tidy per-saccade summary tables, merge saccade metadata into VideoData,
# then save enriched DataFrames (parquet) and QC summaries (CSV).


def build_saccade_summary(video_key):
    """Create a tidy per-saccade summary table for the specified video."""
    results = saccade_results.get(video_key)
    if results is None:
        return pd.DataFrame()

    direction_map = get_direction_map_for_video(video_key)
    summary_parts = []
    for direction, df in (
        ("upward", results.get("upward_saccades_df")),
        ("downward", results.get("downward_saccades_df")),
    ):
        if df is None or df.empty:
            continue
        temp = df.copy()
        temp["direction"] = direction
        temp["direction_label"] = direction_map.get(direction, direction)
        summary_parts.append(temp)

    if not summary_parts:
        return pd.DataFrame()

    summary = pd.concat(summary_parts, ignore_index=True)
    summary = summary.sort_values(["start_time", "time"]).reset_index(drop=True)
    summary.insert(0, "saccade_id", np.arange(1, len(summary) + 1, dtype=int))
    summary["video_key"] = video_key
    summary["eye"] = get_eye_code_for_video(video_key)

    # Normalise frame/index columns
    summary["merge_frame_idx"] = pd.to_numeric(
        summary.get("start_frame_idx"), errors="coerce"
    ).astype("Int64")
    for col_name in [
        "start_frame_idx",
        "peak_frame_idx",
        "end_frame_idx",
    ]:
        if col_name in summary.columns:
            summary[col_name] = pd.to_numeric(
                summary[col_name], errors="coerce"
            ).astype("Int64")

    rename_map = {
        "time": "saccade_peak_time",
        "velocity": "saccade_peak_velocity",
        "start_time": "saccade_start_time",
        "end_time": "saccade_end_time",
        "duration": "saccade_duration",
        "start_position": "saccade_start_position",
        "end_position": "saccade_end_position",
        "amplitude": "saccade_amplitude",
        "displacement": "saccade_displacement",
        "start_frame_idx": "saccade_start_frame_idx",
        "peak_frame_idx": "saccade_peak_frame_idx",
        "end_frame_idx": "saccade_end_frame_idx",
    }
    summary = summary.rename(
        columns={k: v for k, v in rename_map.items() if k in summary.columns}
    )

    preferred_order = [
        "saccade_id",
        "video_key",
        "eye",
        "direction",
        "direction_label",
        "saccade_start_time",
        "saccade_end_time",
        "saccade_peak_time",
        "saccade_start_frame_idx",
        "saccade_peak_frame_idx",
        "saccade_end_frame_idx",
        "saccade_peak_velocity",
        "saccade_amplitude",
        "saccade_displacement",
        "saccade_duration",
        "saccade_start_position",
        "saccade_end_position",
        "saccade_type",
        "bout_id",
        "bout_size",
        "pre_saccade_mean_velocity",
        "pre_saccade_position_drift",
        "post_saccade_position_variance",
        "post_saccade_position_change",
        "merge_frame_idx",
    ]
    remaining_cols = [col for col in summary.columns if col not in preferred_order]
    summary = summary[preferred_order + remaining_cols]
    return summary


summary_tables = {}


def merge_summary_into_video(video_df, summary_df):
    """Merge tidy saccade summary columns into the per-frame VideoData DataFrame."""
    if summary_df is None or summary_df.empty:
        return video_df

    summary_columns = [col for col in summary_df.columns if col != "merge_frame_idx"]
    columns_to_drop = [col for col in summary_columns if col in video_df.columns]
    if columns_to_drop:
        video_df = video_df.drop(columns=columns_to_drop)

    summary_for_join = summary_df.set_index("merge_frame_idx")
    summary_for_join = summary_for_join.drop(columns=["merge_frame_idx"], errors="ignore")

    merged = (
        video_df.set_index("frame_idx")
        .join(summary_for_join, how="left")
        .reset_index()
        .rename(columns={"index": "frame_idx"})
    )
    return merged


if "VideoData1" in globals() and "VideoData1" in saccade_results:

    summary_v1 = build_saccade_summary("VideoData1")
    if not summary_v1.empty:
        VideoData1 = merge_summary_into_video(VideoData1, summary_v1)
        summary_tables["VideoData1"] = summary_v1.drop(columns=["merge_frame_idx"])
    else:
        summary_tables["VideoData1"] = summary_v1

if "VideoData2" in globals() and "VideoData2" in saccade_results:
    summary_v2 = build_saccade_summary("VideoData2")
    if not summary_v2.empty:
        VideoData2 = merge_summary_into_video(VideoData2, summary_v2)
        summary_tables["VideoData2"] = summary_v2.drop(columns=["merge_frame_idx"])
    else:
        summary_tables["VideoData2"] = summary_v2

# Save enriched VideoData tables as parquet and tidy summaries as CSV for QC
downsampled_output_dir = save_path / "downsampled_data"
downsampled_output_dir.mkdir(parents=True, exist_ok=True)

if "VideoData1" in globals():
    video_dir1 = qc_debug_dir / "Video_Sleap_Data1"
    if debug:
        video_dir1.mkdir(parents=True, exist_ok=True)
    summary_output_path1 = downsampled_output_dir / "VideoData1_saccade_summary.csv"

    drop_for_export = [col for col in SACCADE_EXPORT_DROP_COLUMNS if col in VideoData1.columns]
    VideoData1_export = VideoData1.drop(columns=drop_for_export)

    VideoData1_export_renamed = (
        VideoData1_export.rename(columns={"Ellipse.Diameter.Filt": "Pupil.Diameter"})
        if "Ellipse.Diameter.Filt" in VideoData1_export.columns
        else VideoData1_export
    )

    if debug:
        VideoData1_full_indexed = set_aeon_index(VideoData1_export_renamed)
        append_aeon_time_column(VideoData1_full_indexed).to_csv(
            video_dir1 / "Video_Sleap_Data1_1904-01-01T00-00-00.csv", index=False
        )

    drop_for_resample = [col for col in RESAMPLED_DROP_COLUMNS if col in VideoData1_export.columns]
    VideoData1_resampled_input = VideoData1_export.drop(columns=drop_for_resample)
    VideoData1_resampled = resample_video_dataframe(VideoData1_resampled_input, "eye1")
    if "Ellipse.Diameter.Filt" in VideoData1_resampled.columns:
        VideoData1_resampled = VideoData1_resampled.rename(columns={"Ellipse.Diameter.Filt": "Pupil.Diameter"})
    VideoData1_resampled_indexed = set_aeon_index(VideoData1_resampled)
    if debug:
        append_aeon_time_column(VideoData1_resampled_indexed).to_csv(
            video_dir1 / "Video_Sleap_Data1_1904-01-01T00-00-00_resampled.csv", index=False
        )
    downsampled_video1_path = downsampled_output_dir / "VideoData1_resampled.parquet"
    VideoData1_resampled_indexed.to_parquet(
        downsampled_video1_path, engine="pyarrow", compression="snappy"
    )
    print(f"✅ Saved VideoData1 resampled parquet to {downsampled_video1_path}")

    summary_table = summary_tables.get("VideoData1")
    if summary_table is not None and not summary_table.empty:
        summary_export = summary_table.copy()
        summary_export["aeon_time"] = summary_export["saccade_peak_time"].apply(api.aeon)
        summary_export.to_csv(summary_output_path1, index=False)

if "VideoData2" in globals():
    video_dir2 = qc_debug_dir / "Video_Sleap_Data2"
    if debug:
        video_dir2.mkdir(parents=True, exist_ok=True)
    summary_output_path2 = downsampled_output_dir / "VideoData2_saccade_summary.csv"

    drop_for_export = [col for col in SACCADE_EXPORT_DROP_COLUMNS if col in VideoData2.columns]
    VideoData2_export = VideoData2.drop(columns=drop_for_export)

    VideoData2_export_renamed = (
        VideoData2_export.rename(columns={"Ellipse.Diameter.Filt": "Pupil.Diameter"})
        if "Ellipse.Diameter.Filt" in VideoData2_export.columns
        else VideoData2_export
    )

    if debug:
        VideoData2_full_indexed = set_aeon_index(VideoData2_export_renamed)
        append_aeon_time_column(VideoData2_full_indexed).to_csv(
            video_dir2 / "Video_Sleap_Data2_1904-01-01T00-00-00.csv", index=False
        )

    drop_for_resample = [col for col in RESAMPLED_DROP_COLUMNS if col in VideoData2_export.columns]
    VideoData2_resampled_input = VideoData2_export.drop(columns=drop_for_resample)
    VideoData2_resampled = resample_video_dataframe(VideoData2_resampled_input, "eye2")
    if "Ellipse.Diameter.Filt" in VideoData2_resampled.columns:
        VideoData2_resampled = VideoData2_resampled.rename(columns={"Ellipse.Diameter.Filt": "Pupil.Diameter"})
    VideoData2_resampled_indexed = set_aeon_index(VideoData2_resampled)
    if debug:
        append_aeon_time_column(VideoData2_resampled_indexed).to_csv(
            video_dir2 / "Video_Sleap_Data2_1904-01-01T00-00-00_resampled.csv", index=False
        )
    downsampled_video2_path = downsampled_output_dir / "VideoData2_resampled.parquet"
    VideoData2_resampled_indexed.to_parquet(
        downsampled_video2_path, engine="pyarrow", compression="snappy"
    )
    print(f"✅ Saved VideoData2 resampled parquet to {downsampled_video2_path}")

    summary_table = summary_tables.get("VideoData2")
    if summary_table is not None and not summary_table.empty:
        summary_export = summary_table.copy()
        summary_export["aeon_time"] = summary_export["saccade_peak_time"].apply(api.aeon)
        summary_export.to_csv(summary_output_path2, index=False)

In [ ]:
# VISUALIZE ALL SACCADES - SIDE BY SIDE
# -------------------------------------------------------------------------------
# Plot all upward and downward saccades aligned by time with position and velocity traces
# Using extracted visualization function from sleap.visualization

if plot_saccade_detection_QC:
    plot_all_saccades_overlay(
        saccade_results=saccade_results,
        video_labels=VIDEO_LABELS,
        video1_eye=video1_eye,
        video2_eye=video2_eye,
        pre_saccade_window_time=pre_saccade_window_time,
        post_saccade_window_time=post_saccade_window_time,
        debug=debug,
        show_plot=True,
    )

In [ ]:
# SACCADE AMPLITUDE QC VISUALIZATION
# -------------------------------------------------------------------------------
# 1. Distribution of saccade amplitudes
# 2. Correlation between saccade amplitude and duration
# 3. Peri-saccade segments colored by amplitude (outlier detection)
# Using extracted visualization function from sleap.visualization

if plot_saccade_detection_QC:
    plot_saccade_amplitude_qc(
        saccade_results=saccade_results,
        video_labels=VIDEO_LABELS,
        video1_eye=video1_eye,
        video2_eye=video2_eye,
        debug=debug,
        show_plot=True,
    )

In [ ]:
# DEBUG: BASELINING DIAGNOSTICS
# -------------------------------------------------------------------------------
# Plot distribution of baseline window values before and after baselining
# This helps diagnose why some segments might not be baselined correctly

if debug:
    for video_key, res in saccade_results.items():
        dir_map = get_direction_map_for_video(video_key)
        label_up = dir_map["upward"]
        label_down = dir_map["downward"]

        peri_saccades = res["peri_saccades"]

        if len(peri_saccades) == 0:
            print(
                f"\n⚠️  No saccades found for {get_eye_label(video_key)}, skipping baselining diagnostics"
            )
            continue

        # Extract baseline window statistics for each segment
        baseline_values = []  # What was subtracted
        # Mean position in baseline window BEFORE baselining
        baseline_window_means_before = []
        # Mean position in baseline window AFTER baselining
        baseline_window_means_after = []
        baseline_window_counts = []  # Number of points in baseline window
        segment_directions = []
        segment_ids = []

        # Get baseline window parameters from the function call (use defaults
        # if not available)
        baseline_window_start_time = (
            -0.1
            if "baseline_window_start_time" not in globals()
            else baseline_window_start_time
        )
        baseline_window_end_time = (
            -0.02
            if "baseline_window_end_time" not in globals()
            else baseline_window_end_time
        )

        for seg in peri_saccades:
            seg_id = (
                seg["saccade_id"].iloc[0]
                if "saccade_id" in seg.columns
                else len(baseline_values)
            )
            direction = (
                seg["saccade_direction"].iloc[0]
                if "saccade_direction" in seg.columns
                else "unknown"
            )

            # Get baseline value that was used
            if "baseline_value" in seg.columns:
                baseline_val = seg["baseline_value"].iloc[0]
            else:
                baseline_val = np.nan

            # Find baseline window points (before threshold crossing)
            baseline_mask = (
                (seg["Time_rel_threshold"] >= baseline_window_start_time)
                & (seg["Time_rel_threshold"] <= baseline_window_end_time)
                & (seg["Time_rel_threshold"] < 0)  # Pre-threshold only
            )

            # Get original position values (reconstruct if needed)
            if "X_raw" in seg.columns:
                original_pos_col = "X_raw"
            elif "X_smooth" in seg.columns:
                original_pos_col = "X_smooth"
            else:
                original_pos_col = None

            # Calculate mean in baseline window BEFORE baselining
            if original_pos_col is not None:
                baseline_window_original = seg.loc[
                    baseline_mask, original_pos_col
                ].dropna()
                if len(baseline_window_original) > 0:
                    mean_before = baseline_window_original.mean()
                else:
                    mean_before = np.nan
            else:
                # Reconstruct: original = baselined + baseline_value
                if "X_smooth_baselined" in seg.columns and not pd.isna(baseline_val):
                    baseline_window_baselined = seg.loc[
                        baseline_mask, "X_smooth_baselined"
                    ].dropna()
                    if len(baseline_window_baselined) > 0:
                        mean_before = baseline_window_baselined.mean() + baseline_val
                    else:
                        mean_before = np.nan
                else:
                    mean_before = np.nan

            # Calculate mean in baseline window AFTER baselining
            if "X_smooth_baselined" in seg.columns:
                baseline_window_baselined = seg.loc[
                    baseline_mask, "X_smooth_baselined"
                ].dropna()
                if len(baseline_window_baselined) > 0:
                    mean_after = baseline_window_baselined.mean()
                    n_points = len(baseline_window_baselined)
                else:
                    mean_after = np.nan
                    n_points = 0
            else:
                mean_after = np.nan
                n_points = 0

            baseline_values.append(baseline_val)
            baseline_window_means_before.append(mean_before)
            baseline_window_means_after.append(mean_after)
            baseline_window_counts.append(n_points)
            segment_directions.append(direction)
            segment_ids.append(seg_id)

        # Convert to arrays for easier manipulation
        baseline_values = np.array(baseline_values)
        baseline_window_means_before = np.array(baseline_window_means_before)
        baseline_window_means_after = np.array(baseline_window_means_after)
        baseline_window_counts = np.array(baseline_window_counts)
        segment_directions = np.array(segment_directions)

        # Create diagnostic figure
        fig_baseline = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Baseline Values (What Was Subtracted)",
                "Baseline Window Mean - BEFORE Baselining",
                "Baseline Window Mean - AFTER Baselining",
                "Baseline Window Point Counts",
            ),
            vertical_spacing=0.15,
            horizontal_spacing=0.12,
        )

        # Plot 1: Baseline values distribution
        for direction in ["upward", "downward"]:
            mask = segment_directions == direction
            if mask.sum() > 0:
                label = label_up if direction == "upward" else label_down
                color = "green" if direction == "upward" else "purple"
                fig_baseline.add_trace(
                    go.Histogram(
                        x=baseline_values[mask],
                        nbinsx=50,
                        name=f"{label}",
                        marker_color=color,
                        opacity=0.6,
                    ),
                    row=1,
                    col=1,
                )

        # Plot 2: Baseline window mean BEFORE baselining
        for direction in ["upward", "downward"]:
            mask = segment_directions == direction
            if mask.sum() > 0:
                label = label_up if direction == "upward" else label_down
                color = "green" if direction == "upward" else "purple"
                valid_mask = mask & ~np.isnan(baseline_window_means_before)
                if valid_mask.sum() > 0:
                    fig_baseline.add_trace(
                        go.Histogram(
                            x=baseline_window_means_before[valid_mask],
                            nbinsx=50,
                            name=f"{label}",
                            marker_color=color,
                            opacity=0.6,
                            showlegend=False,
                        ),
                        row=1,
                        col=2,
                    )

        # Plot 3: Baseline window mean AFTER baselining (should be ~0)
        for direction in ["upward", "downward"]:
            mask = segment_directions == direction
            if mask.sum() > 0:
                label = label_up if direction == "upward" else label_down
                color = "green" if direction == "upward" else "purple"
                valid_mask = mask & ~np.isnan(baseline_window_means_after)
                if valid_mask.sum() > 0:
                    fig_baseline.add_trace(
                        go.Histogram(
                            x=baseline_window_means_after[valid_mask],
                            nbinsx=50,
                            name=f"{label}",
                            marker_color=color,
                            opacity=0.6,
                            showlegend=False,
                        ),
                        row=2,
                        col=1,
                    )

        # Plot 4: Baseline window point counts
        for direction in ["upward", "downward"]:
            mask = segment_directions == direction
            if mask.sum() > 0:
                label = label_up if direction == "upward" else label_down
                color = "green" if direction == "upward" else "purple"
                fig_baseline.add_trace(
                    go.Histogram(
                        x=baseline_window_counts[mask],
                        nbinsx=20,
                        name=f"{label}",
                        marker_color=color,
                        opacity=0.6,
                        showlegend=False,
                    ),
                    row=2,
                    col=2,
                )

        # Add vertical line at 0 for "AFTER baselining" plot
        fig_baseline.add_vline(
            x=0,
            line_dash="dash",
            line_color="red",
            line_width=2,
            opacity=0.7,
            annotation_text="Expected: 0",
            row=2,
            col=1,
        )

        # Update layout
        fig_baseline.update_layout(
            title_text=f"Baselining Diagnostics: {get_eye_label(video_key)}<br><sub>After baselining, baseline window mean should be ~0</sub>",
            height=800,
            showlegend=True,
            legend=dict(x=0.02, y=0.98),
        )

        # Update axes labels
        fig_baseline.update_xaxes(title_text="Baseline Value (px)", row=1, col=1)
        fig_baseline.update_xaxes(
            title_text="Mean Position in Baseline Window (px)", row=1, col=2
        )
        fig_baseline.update_xaxes(
            title_text="Mean Position in Baseline Window (px)", row=2, col=1
        )
        fig_baseline.update_xaxes(title_text="Number of Points", row=2, col=2)
        fig_baseline.update_yaxes(title_text="Count", row=1, col=1)
        fig_baseline.update_yaxes(title_text="Count", row=1, col=2)
        fig_baseline.update_yaxes(title_text="Count", row=2, col=1)
        fig_baseline.update_yaxes(title_text="Count", row=2, col=2)

        fig_baseline.show()

        # Print statistics
        print(f"\n{'=' * 80}")
        print(f"BASELINING DIAGNOSTICS: {get_eye_label(video_key)}")
        print(f"{'=' * 80}")

        for direction in ["upward", "downward"]:
            mask = segment_directions == direction
            label = label_up if direction == "upward" else label_down

            if mask.sum() > 0:
                print(f"\n{label} saccades (n={mask.sum()}):")

                # Baseline values
                valid_baseline = baseline_values[mask & ~np.isnan(baseline_values)]
                if len(valid_baseline) > 0:
                    print("  Baseline values (subtracted):")
                    print(f"    Mean: {valid_baseline.mean():.2f} px")
                    print(f"    Std: {valid_baseline.std():.2f} px")
                    print(
                        f"    Range: [{valid_baseline.min():.2f}, {valid_baseline.max():.2f}] px"
                    )
                    print(f"    NaN count: {np.isnan(baseline_values[mask]).sum()}")

                # Baseline window means BEFORE
                valid_before = baseline_window_means_before[
                    mask & ~np.isnan(baseline_window_means_before)
                ]
                if len(valid_before) > 0:
                    print("\n  Baseline window mean BEFORE baselining:")
                    print(f"    Mean: {valid_before.mean():.2f} px")
                    print(f"    Std: {valid_before.std():.2f} px")
                    print(
                        f"    Range: [{valid_before.min():.2f}, {valid_before.max():.2f}] px"
                    )
                    print(
                        f"    NaN count: {np.isnan(baseline_window_means_before[mask]).sum()}"
                    )

                # Baseline window means AFTER (should be ~0)
                valid_after = baseline_window_means_after[
                    mask & ~np.isnan(baseline_window_means_after)
                ]
                if len(valid_after) > 0:
                    print("\n  Baseline window mean AFTER baselining (should be ~0):")
                    print(f"    Mean: {valid_after.mean():.2f} px")
                    print(f"    Std: {valid_after.std():.2f} px")
                    print(
                        f"    Range: [{valid_after.min():.2f}, {valid_after.max():.2f}] px"
                    )
                    print(
                        f"    NaN count: {np.isnan(baseline_window_means_after[mask]).sum()}"
                    )

                    # Count segments that are NOT properly baselined (mean >
                    # 1px away from 0)
                    not_baselined = np.abs(valid_after) > 1.0
                    if not_baselined.sum() > 0:
                        print(
                            f"\n  ⚠️  WARNING: {not_baselined.sum()} segments NOT properly baselined (|mean| > 1px)"
                        )
                        print(
                            f"    Mean values for non-baselined segments: {valid_after[not_baselined]}"
                        )

                    # Show distribution of baseline window means (should all be
                    # ~0)
                    print(
                        "\n  Distribution of baseline window means AFTER baselining:"
                    )
                    print(
                        f"    Segments with |mean| < 0.1 px: {(np.abs(valid_after) < 0.1).sum()}"
                    )
                    print(
                        f"    Segments with 0.1 <= |mean| < 0.5 px: {((np.abs(valid_after) >= 0.1) & (np.abs(valid_after) < 0.5)).sum()}"
                    )
                    print(
                        f"    Segments with 0.5 <= |mean| < 1.0 px: {((np.abs(valid_after) >= 0.5) & (np.abs(valid_after) < 1.0)).sum()}"
                    )
                    print(
                        f"    Segments with |mean| >= 1.0 px: {(np.abs(valid_after) >= 1.0).sum()}"
                    )

                    # Show worst offenders
                    worst_indices = np.argsort(np.abs(valid_after))[
                        -10:
                    ]  # Top 10 worst
                    if len(worst_indices) > 0:
                        print(
                            "\n  Top 10 segments with largest |baseline window mean|:"
                        )
                        for i, idx in enumerate(
                            worst_indices[::-1]
                        ):  # Reverse to show worst first
                            seg_idx = np.where(mask)[0][idx]
                            seg_id = segment_ids[seg_idx]
                            mean_val = valid_after[idx]
                            baseline_val = baseline_values[mask][idx]
                            print(
                                f"    Segment {seg_id}: baseline_window_mean={mean_val:.3f} px, baseline_value={baseline_val:.3f} px"
                            )

                    # CRITICAL CHECK: Verify that baseline_value actually equals baseline_window_mean_before
                    # This checks if baselining logic is correct
                    valid_before_check = baseline_window_means_before[
                        mask & ~np.isnan(baseline_window_means_before)
                    ]
                    if len(valid_before_check) > 0 and len(valid_baseline) > 0:
                        # Check if baseline_value matches
                        # baseline_window_mean_before (should be identical)
                        # Allow small floating point differences
                        mismatches = np.abs(valid_baseline - valid_before_check) > 0.01
                        if mismatches.sum() > 0:
                            print(
                                f"\n  ⚠️  CRITICAL: {mismatches.sum()} segments where baseline_value != baseline_window_mean_before"
                            )
                            print("    This indicates a bug in baselining logic!")
                            for i in np.where(mismatches)[0][:5]:  # Show first 5
                                print(
                                    f"      Segment {i}: baseline_value={valid_baseline[i]:.3f}, baseline_window_mean_before={valid_before_check[i]:.3f}"
                                )
                        else:
                            print(
                                "\n  ✅ Baselining logic check: baseline_value matches baseline_window_mean_before for all segments"
                            )

                # Baseline window point counts
                valid_counts = baseline_window_counts[
                    mask & (baseline_window_counts > 0)
                ]
                if len(valid_counts) > 0:
                    print("\n  Baseline window point counts:")
                    print(f"    Mean: {valid_counts.mean():.1f} points")
                    print(
                        f"    Range: [{valid_counts.min()}, {valid_counts.max()}] points"
                    )
                    print(
                        f"    Segments with 0 points in baseline window: {(baseline_window_counts[mask] == 0).sum()}"
                    )

        print(f"\n{'=' * 80}\n")

        # Additional diagnostic: Check if baselining is actually applied to segments
        # Compare original vs baselined values at the start of segments (before
        # threshold)
        print(f"\n{'=' * 80}")
        print("ADDITIONAL BASELINING CHECK: Comparing segment start positions")
        print(f"{'=' * 80}")

        for direction in ["upward", "downward"]:
            mask = segment_directions == direction
            label = label_up if direction == "upward" else label_down

            if mask.sum() > 0:
                print(f"\n{label} saccades:")

                # Check first few points of each segment (should be ~0 after
                # baselining)
                segment_start_means_original = []
                segment_start_means_baselined = []

                for i, seg in enumerate(peri_saccades):
                    if segment_directions[i] != direction:
                        continue

                    # Get first 5 points before threshold (if available)
                    pre_threshold_mask = seg["Time_rel_threshold"] < 0
                    pre_threshold_seg = seg.loc[pre_threshold_mask].head(5)

                    if len(pre_threshold_seg) > 0:
                        # Get original position
                        if "X_raw" in seg.columns:
                            orig_col = "X_raw"
                        elif "X_smooth" in seg.columns:
                            orig_col = "X_smooth"
                        else:
                            orig_col = None

                        if orig_col is not None:
                            orig_mean = pre_threshold_seg[orig_col].mean()
                            segment_start_means_original.append(orig_mean)

                        # Get baselined position
                        if "X_smooth_baselined" in seg.columns:
                            baselined_mean = pre_threshold_seg[
                                "X_smooth_baselined"
                            ].mean()
                            segment_start_means_baselined.append(baselined_mean)

                if len(segment_start_means_baselined) > 0:
                    segment_start_means_baselined = np.array(
                        segment_start_means_baselined
                    )
                    print(
                        "  Mean position at segment start (first 5 pre-threshold points) AFTER baselining:"
                    )
                    print(f"    Mean: {segment_start_means_baselined.mean():.3f} px")
                    print(f"    Std: {segment_start_means_baselined.std():.3f} px")
                    print(
                        f"    Range: [{segment_start_means_baselined.min():.3f}, {segment_start_means_baselined.max():.3f}] px"
                    )
                    print(
                        f"    Segments with |mean| > 1 px: {(np.abs(segment_start_means_baselined) > 1.0).sum()}"
                    )
                    print(
                        f"    Segments with |mean| > 5 px: {(np.abs(segment_start_means_baselined) > 5.0).sum()}"
                    )

                    # Show worst offenders
                    worst_indices = np.argsort(np.abs(segment_start_means_baselined))[
                        -10:
                    ]
                    if len(worst_indices) > 0:
                        print(
                            "\n  Top 10 segments with largest |start position mean| AFTER baselining:"
                        )
                        for idx in worst_indices[::-1]:
                            print(
                                f"    Segment {idx}: start_mean={segment_start_means_baselined[idx]:.3f} px"
                            )

        print(f"\n{'=' * 80}\n")

In [ ]:
# VISUALIZE DETECTED SACCADES (Adaptive Method)
# -------------------------------------------------------------------------------
# Create overlay plot showing detected saccades with duration lines and
# peak arrows
if plot_saccade_detection_QC:
    for video_key, res in saccade_results.items():
        dir_map = get_direction_map_for_video(video_key)
        label_up = dir_map["upward"]
        label_down = dir_map["downward"]

        upward_saccades_df = res["upward_saccades_df"]
        downward_saccades_df = res["downward_saccades_df"]
        peri_saccades = res["peri_saccades"]
        upward_segments = res["upward_segments"]
        downward_segments = res["downward_segments"]
        # Any other variables you need...

        # Optional: Find 5-minute window with highest saccade density
        plot_5min_window = (
            True  # Set to True to plot only highest density 5-minute window
        )
        window_duration = 300  # 5 minutes in seconds

        # Initialize variables for windowing
        best_window_start = None
        best_window_end = None
        best_window_count = 0

        if plot_5min_window and (
            len(upward_saccades_df) > 0 or len(downward_saccades_df) > 0
        ):
            # Combine all saccades and find time window with highest density
            all_saccade_times = []
            if len(upward_saccades_df) > 0:
                all_saccade_times.extend(upward_saccades_df["time"].values)
            if len(downward_saccades_df) > 0:
                all_saccade_times.extend(downward_saccades_df["time"].values)

            if len(all_saccade_times) > 0:
                all_saccade_times = np.array(all_saccade_times)
                time_min = all_saccade_times.min()
                time_max = all_saccade_times.max()

                # Slide window and count saccades in each window
                best_window_start = time_min
                best_window_count = 0
                step_size = 10  # Check every 10 seconds

                for window_start in np.arange(
                    time_min, time_max - window_duration + step_size, step_size
                ):
                    window_end = window_start + window_duration
                    count = np.sum(
                        (all_saccade_times >= window_start)
                        & (all_saccade_times <= window_end)
                    )
                    if count > best_window_count:
                        best_window_count = count
                        best_window_start = window_start

                best_window_end = best_window_start + window_duration

                # Filter data to this window
                df_window = res["df"][
                    (res["df"]["Seconds"] >= best_window_start)
                    & (res["df"]["Seconds"] <= best_window_end)
                ].copy()
                upward_saccades_df_window = upward_saccades_df[
                    (upward_saccades_df["time"] >= best_window_start)
                    & (upward_saccades_df["time"] <= best_window_end)
                ].copy()
                downward_saccades_df_window = downward_saccades_df[
                    (downward_saccades_df["time"] >= best_window_start)
                    & (downward_saccades_df["time"] <= best_window_end)
                ].copy()

                if debug:
                    print(
                        f"\n📊 Highest saccade density window: {best_window_start:.1f}s to {best_window_end:.1f}s"
                    )
                    print(
                        f"   ({best_window_count} saccades in {window_duration / 60:.1f} minutes)"
                    )
                    print(
                        f"   Density: {best_window_count / (window_duration / 60):.1f} saccades/min"
                    )
            else:
                plot_5min_window = False
        else:
            plot_5min_window = False

        # Use windowed data if requested, otherwise use full data
        if plot_5min_window:
            df_plot = df_window
            upward_saccades_df_plot = upward_saccades_df_window
            downward_saccades_df_plot = downward_saccades_df_window
            time_range_text = f" (5-min window: {best_window_start:.1f}-{best_window_end:.1f}s, {best_window_count} saccades)"
        else:
            df_plot = res["df"]
            upward_saccades_df_plot = upward_saccades_df
            downward_saccades_df_plot = downward_saccades_df
            time_range_text = ""

        fig = make_subplots(
            rows=2,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.1,
            subplot_titles=(
                "X Position (px)",
                "Velocity (px/s) with Detected Saccades",
            ),
        )

        # Add smoothed X position to the first subplot
        fig.add_trace(
            go.Scatter(
                x=df_plot["Seconds"],
                y=df_plot["X_smooth"],
                mode="lines",
                name="Smoothed X",
                line=dict(color="blue", width=2),
            ),
            row=1,
            col=1,
        )

        # Add smoothed velocity to the second subplot
        fig.add_trace(
            go.Scatter(
                x=df_plot["Seconds"],
                y=df_plot["vel_x_smooth"],
                mode="lines",
                name="Smoothed Velocity",
                line=dict(color="red", width=2),
            ),
            row=2,
            col=1,
        )

        # Add adaptive threshold lines for reference
        fig.add_hline(
            y=res["vel_thresh"],
            line_dash="dash",
            line_color="green",
            opacity=0.5,
            annotation_text=f"Adaptive threshold (±{res['vel_thresh']:.0f} px/s)",
            row=2,
            col=1,
        )

        fig.add_hline(
            y=-res["vel_thresh"],
            line_dash="dash",
            line_color="green",
            opacity=0.5,
            row=2,
            col=1,
        )

        # Calculate position y-axis range for vertical lines
        pos_max = df_plot["X_smooth"].max()
        pos_min = df_plot["X_smooth"].min()
        pos_range = pos_max - pos_min
        # Add small padding to ensure lines span full visible range
        pos_padding = pos_range * 0.05
        pos_y_min = pos_min - pos_padding
        pos_y_max = pos_max + pos_padding

        # Plot upward saccades (TN) as vertical lines on position trace
        if len(upward_saccades_df_plot) > 0:
            for idx, row in upward_saccades_df_plot.iterrows():
                start_time = row["start_time"]
                end_time = row["end_time"]

                # Use rectangle with thin border to show duration span more efficiently
                # Opacity must be set at shape level or via rgba color, not in
                # line dict
                fig.add_shape(
                    type="rect",
                    x0=start_time,
                    y0=pos_y_min,
                    x1=end_time,
                    y1=pos_y_max,
                    # Light green fill with opacity
                    fillcolor="rgba(0,128,0,0.1)",
                    # Green border with opacity via rgba
                    line=dict(color="rgba(0,128,0,0.3)", width=2),
                    row=1,
                    col=1,
                )

            # Add legend entry for upward saccades
            fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="lines",
                    name=f"{label_up} Saccades",
                    line=dict(
                        color="rgba(0,128,0,0.3)", width=3
                    ),  # Green with opacity via rgba
                ),
                row=1,
                col=1,
            )

        # Plot downward saccades (NT) as vertical lines on position trace
        if len(downward_saccades_df_plot) > 0:
            for idx, row in downward_saccades_df_plot.iterrows():
                start_time = row["start_time"]
                end_time = row["end_time"]

                # Use rectangle with thin border to show duration span more efficiently
                # Opacity must be set at shape level or via rgba color, not in
                # line dict
                fig.add_shape(
                    type="rect",
                    x0=start_time,
                    y0=pos_y_min,
                    x1=end_time,
                    y1=pos_y_max,
                    # Light purple fill with opacity
                    fillcolor="rgba(128,0,128,0.1)",
                    # Purple border with opacity via rgba
                    line=dict(color="rgba(128,0,128,0.3)", width=2),
                    row=1,
                    col=1,
                )

            # Add legend entry for downward saccades
            fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="lines",
                    name=f"{label_down} Saccades",
                    line=dict(
                        color="rgba(128,0,128,0.3)", width=3
                    ),  # Purple with opacity via rgba
                ),
                row=1,
                col=1,
            )

        # Update layout
        fig.update_layout(
            title=f"Detected Saccades ({get_eye_label(video_key)}): Vertical Lines on Position Trace (QC Visualization){time_range_text}",
            height=600,
            showlegend=True,
            legend=dict(x=0.01, y=0.99),
        )

        # Update axes
        fig.update_xaxes(title_text="Time (s)", row=2, col=1)
        fig.update_yaxes(title_text="X Position (px)", row=1, col=1)
        fig.update_yaxes(title_text="Velocity (px/s)", row=2, col=1)

        fig.show()

In [ ]:
# ADAPTIVE THRESHOLD DIAGNOSTIC PLOTS (only if debug=True)
# -------------------------------------------------------------------------------
# Plot distributions of classification features to help determine
# meaningful adaptive thresholds
if debug and len(saccade_results) > 0:
    print("\n📊 Generating adaptive threshold diagnostic plots...")

    for video_key, res in saccade_results.items():
        all_saccades_df = res.get("all_saccades_df", pd.DataFrame())

        if len(all_saccades_df) == 0:
            print(
                f"⚠️  No saccades found for {get_eye_label(video_key)}, skipping diagnostic plots"
            )
            continue

        # Filter out NaN values for plotting
        pre_vel = all_saccades_df["pre_saccade_mean_velocity"].dropna()
        pre_drift = all_saccades_df["pre_saccade_position_drift"].dropna()
        post_var = all_saccades_df["post_saccade_position_variance"].dropna()
        post_change = all_saccades_df["post_saccade_position_change"].dropna()
        amplitude = all_saccades_df["amplitude"].dropna()

        # Calculate post_change / amplitude ratio (for percentage threshold visualization)
        # Align by index to ensure matching
        aligned_indices = post_change.index.intersection(amplitude.index)
        post_change_aligned = post_change.loc[aligned_indices]
        amplitude_aligned = amplitude.loc[aligned_indices]
        # Convert to percentage
        post_change_ratio = (post_change_aligned / amplitude_aligned) * 100

        # Calculate current thresholds for visualization
        if use_adaptive_thresholds:
            # Calculate adaptive thresholds from current data
            if len(pre_vel) >= 3:
                current_pre_vel_threshold = np.percentile(
                    pre_vel, adaptive_percentile_pre_velocity
                )
            else:
                current_pre_vel_threshold = pre_saccade_velocity_threshold

            if len(pre_drift) >= 3:
                current_pre_drift_threshold = np.percentile(
                    pre_drift, adaptive_percentile_pre_drift
                )
            else:
                current_pre_drift_threshold = pre_saccade_drift_threshold

            if len(post_var) >= 3:
                current_post_var_threshold = np.percentile(
                    post_var, adaptive_percentile_post_variance
                )
            else:
                current_post_var_threshold = post_saccade_variance_threshold
        else:
            # Use fixed thresholds
            current_pre_vel_threshold = pre_saccade_velocity_threshold
            current_pre_drift_threshold = pre_saccade_drift_threshold
            current_post_var_threshold = post_saccade_variance_threshold

        # Create figure with 2x2 subplots
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(
            f"Adaptive Threshold Diagnostic Plots: {get_eye_label(video_key)}\n"
            f"(n={len(all_saccades_df)} saccades)",
            fontsize=14,
            fontweight="bold",
        )

        # Plot 1: Pre-saccade mean velocity
        ax = axes[0, 0]
        if len(pre_vel) > 0:
            ax.hist(pre_vel, bins=50, alpha=0.7, color="skyblue", edgecolor="black")
            ax.axvline(
                current_pre_vel_threshold,
                color="red",
                linestyle="--",
                linewidth=2,
                label=f"Threshold: {current_pre_vel_threshold:.2f} px/s",
            )
            if use_adaptive_thresholds:
                ax.axvline(
                    np.percentile(pre_vel, 50),
                    color="gray",
                    linestyle=":",
                    linewidth=1,
                    label=f"Median: {np.percentile(pre_vel, 50):.2f} px/s",
                )
                ax.axvline(
                    np.percentile(pre_vel, 75),
                    color="orange",
                    linestyle=":",
                    linewidth=1,
                    label=f"75th: {np.percentile(pre_vel, 75):.2f} px/s",
                )
            ax.set_xlabel("Pre-saccade Mean Velocity (px/s)")
            ax.set_ylabel("Count")
            ax.set_title(
                f"Pre-saccade Velocity Distribution\n"
                f"{'Adaptive' if use_adaptive_thresholds else 'Fixed'} threshold at "
                f"{adaptive_percentile_pre_velocity if use_adaptive_thresholds else 'fixed'}th percentile"
            )
            ax.legend()
            ax.grid(True, alpha=0.3)

        # Plot 2: Pre-saccade position drift
        ax = axes[0, 1]
        if len(pre_drift) > 0:
            ax.hist(
                pre_drift, bins=50, alpha=0.7, color="lightgreen", edgecolor="black"
            )
            ax.axvline(
                current_pre_drift_threshold,
                color="red",
                linestyle="--",
                linewidth=2,
                label=f"Threshold: {current_pre_drift_threshold:.2f} px",
            )
            if use_adaptive_thresholds:
                ax.axvline(
                    np.percentile(pre_drift, 50),
                    color="gray",
                    linestyle=":",
                    linewidth=1,
                    label=f"Median: {np.percentile(pre_drift, 50):.2f} px",
                )
                ax.axvline(
                    np.percentile(pre_drift, 75),
                    color="orange",
                    linestyle=":",
                    linewidth=1,
                    label=f"75th: {np.percentile(pre_drift, 75):.2f} px",
                )
            ax.set_xlabel("Pre-saccade Position Drift (px)")
            ax.set_ylabel("Count")
            ax.set_title(
                f"Pre-saccade Drift Distribution\n"
                f"{'Adaptive' if use_adaptive_thresholds else 'Fixed'} threshold at "
                f"{adaptive_percentile_pre_drift if use_adaptive_thresholds else 'fixed'}th percentile"
            )
            ax.legend()
            ax.grid(True, alpha=0.3)

        # Plot 3: Post-saccade position variance
        ax = axes[1, 0]
        if len(post_var) > 0:
            ax.hist(post_var, bins=50, alpha=0.7, color="plum", edgecolor="black")
            ax.axvline(
                current_post_var_threshold,
                color="red",
                linestyle="--",
                linewidth=2,
                label=f"Threshold: {current_post_var_threshold:.2f} px²",
            )
            if use_adaptive_thresholds:
                ax.axvline(
                    np.percentile(post_var, 25),
                    color="orange",
                    linestyle=":",
                    linewidth=1,
                    label=f"25th: {np.percentile(post_var, 25):.2f} px²",
                )
                ax.axvline(
                    np.percentile(post_var, 50),
                    color="gray",
                    linestyle=":",
                    linewidth=1,
                    label=f"Median: {np.percentile(post_var, 50):.2f} px²",
                )
            ax.set_xlabel("Post-saccade Position Variance (px²)")
            ax.set_ylabel("Count")
            ax.set_title(
                f"Post-saccade Variance Distribution\n"
                f"{'Adaptive' if use_adaptive_thresholds else 'Fixed'} threshold at "
                f"{adaptive_percentile_post_variance if use_adaptive_thresholds else 'fixed'}th percentile"
            )
            ax.legend()
            ax.grid(True, alpha=0.3)

        # Plot 4: Post-saccade position change (as percentage of amplitude)
        ax = axes[1, 1]
        if len(post_change_ratio) > 0:
            ax.hist(
                post_change_ratio, bins=50, alpha=0.7, color="salmon", edgecolor="black"
            )
            ax.axvline(
                post_saccade_position_change_threshold_percent,
                color="red",
                linestyle="--",
                linewidth=2,
                label=f"Threshold: {post_saccade_position_change_threshold_percent:.1f}%",
            )
            ax.axvline(
                np.percentile(post_change_ratio, 50),
                color="gray",
                linestyle=":",
                linewidth=1,
                label=f"Median: {np.percentile(post_change_ratio, 50):.1f}%",
            )
            ax.axvline(
                np.percentile(post_change_ratio, 75),
                color="orange",
                linestyle=":",
                linewidth=1,
                label=f"75th: {np.percentile(post_change_ratio, 75):.1f}%",
            )
            ax.set_xlabel("Post-saccade Position Change / Amplitude (%)")
            ax.set_ylabel("Count")
            ax.set_title(
                f"Post-saccade Position Change Ratio\n"
                f"Fixed threshold: {post_saccade_position_change_threshold_percent:.1f}% of amplitude"
            )
            ax.legend()
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Print summary statistics
        print(f"\n📈 Summary Statistics for {get_eye_label(video_key)}:")
        if len(pre_vel) > 0:
            print(
                f"  Pre-saccade velocity: mean={pre_vel.mean():.2f}, median={pre_vel.median():.2f}, "
                f"std={pre_vel.std():.2f} px/s"
            )
        if len(pre_drift) > 0:
            print(
                f"  Pre-saccade drift: mean={pre_drift.mean():.2f}, median={pre_drift.median():.2f}, "
                f"std={pre_drift.std():.2f} px"
            )
        if len(post_var) > 0:
            print(
                f"  Post-saccade variance: mean={post_var.mean():.2f}, median={post_var.median():.2f}, "
                f"std={post_var.std():.2f} px²"
            )
        if len(post_change_ratio) > 0:
            print(
                f"  Post-saccade change ratio: mean={post_change_ratio.mean():.1f}%, "
                f"median={post_change_ratio.median():.1f}%, std={post_change_ratio.std():.1f}%"
            )
        print()

In [ ]:
# VISUALIZE AND ANALYZE SACCADE CLASSIFICATION (Orienting vs Compensatory)
# -------------------------------------------------------------------------------
# Create validation plots and statistical comparisons for saccade
# classification

if plot_saccade_detection_QC:
    for video_key, res in saccade_results.items():
        dir_map = get_direction_map_for_video(video_key)
        label_up = dir_map["upward"]
        label_down = dir_map["downward"]

        all_saccades_df = res.get("all_saccades_df", pd.DataFrame())

        if len(all_saccades_df) == 0:
            print(f"\n⚠️ No saccades found for {get_eye_label(video_key)}")
            continue

        # Check if classification was performed

        if "saccade_type" not in all_saccades_df.columns:
            print(f"\n⚠️ Classification not performed for {get_eye_label(video_key)}")
            continue

        orienting_saccades = all_saccades_df[
            all_saccades_df["saccade_type"] == "orienting"
        ]

        compensatory_saccades = all_saccades_df[
            all_saccades_df["saccade_type"] == "compensatory"
        ]

        print(f"\n{'=' * 80}")

        print(f"CLASSIFICATION ANALYSIS: {get_eye_label(video_key)}")

        print(f"{'=' * 80}")

        # Statistical comparisons

        from scipy import stats

        print("\n📊 Statistical Comparisons:")

        print(f"  Orienting saccades: {len(orienting_saccades)}")

        print(f"  Compensatory saccades: {len(compensatory_saccades)}")

        if len(orienting_saccades) > 0 and len(compensatory_saccades) > 0:
            # Amplitude comparison

            orienting_amps = orienting_saccades["amplitude"].values

            compensatory_amps = compensatory_saccades["amplitude"].values

            amp_stat, amp_p = stats.mannwhitneyu(
                orienting_amps, compensatory_amps, alternative="two-sided"
            )

            print("\n  Amplitude (px):")

            print(
                f"    Orienting: {orienting_amps.mean():.2f} ± {orienting_amps.std():.2f} (median: {np.median(orienting_amps):.2f})"
            )

            print(
                f"    Compensatory: {compensatory_amps.mean():.2f} ± {compensatory_amps.std():.2f} (median: {np.median(compensatory_amps):.2f})"
            )

            print(f"    Mann-Whitney U test: U={amp_stat:.1f}, p={amp_p:.4f}")

            # Duration comparison

            orienting_durs = orienting_saccades["duration"].values

            compensatory_durs = compensatory_saccades["duration"].values

            dur_stat, dur_p = stats.mannwhitneyu(
                orienting_durs, compensatory_durs, alternative="two-sided"
            )

            print("\n  Duration (s):")

            print(
                f"    Orienting: {orienting_durs.mean():.3f} ± {orienting_durs.std():.3f} (median: {np.median(orienting_durs):.3f})"
            )

            print(
                f"    Compensatory: {compensatory_durs.mean():.3f} ± {compensatory_durs.std():.3f} (median: {np.median(compensatory_durs):.3f})"
            )

            print(f"    Mann-Whitney U test: U={dur_stat:.1f}, p={dur_p:.4f}")

            # Pre-saccade velocity comparison

            orienting_pre_vel = orienting_saccades["pre_saccade_mean_velocity"].values

            compensatory_pre_vel = compensatory_saccades[
                "pre_saccade_mean_velocity"
            ].values

            pre_vel_stat, pre_vel_p = stats.mannwhitneyu(
                orienting_pre_vel, compensatory_pre_vel, alternative="two-sided"
            )

            print("\n  Pre-saccade velocity (px/s):")

            print(
                f"    Orienting: {orienting_pre_vel.mean():.2f} ± {orienting_pre_vel.std():.2f} (median: {np.median(orienting_pre_vel):.2f})"
            )

            print(
                f"    Compensatory: {compensatory_pre_vel.mean():.2f} ± {compensatory_pre_vel.std():.2f} (median: {np.median(compensatory_pre_vel):.2f})"
            )

            print(f"    Mann-Whitney U test: U={pre_vel_stat:.1f}, p={pre_vel_p:.4f}")

            # Pre-saccade drift comparison

            orienting_pre_drift = orienting_saccades[
                "pre_saccade_position_drift"
            ].values

            compensatory_pre_drift = compensatory_saccades[
                "pre_saccade_position_drift"
            ].values

            pre_drift_stat, pre_drift_p = stats.mannwhitneyu(
                orienting_pre_drift, compensatory_pre_drift, alternative="two-sided"
            )

            print("\n  Pre-saccade position drift (px):")

            print(
                f"    Orienting: {orienting_pre_drift.mean():.2f} ± {orienting_pre_drift.std():.2f} (median: {np.median(orienting_pre_drift):.2f})"
            )

            print(
                f"    Compensatory: {compensatory_pre_drift.mean():.2f} ± {compensatory_pre_drift.std():.2f} (median: {np.median(compensatory_pre_drift):.2f})"
            )

            print(
                f"    Mann-Whitney U test: U={pre_drift_stat:.1f}, p={pre_drift_p:.4f}"
            )

            # Post-saccade variance comparison

            orienting_post_var = orienting_saccades[
                "post_saccade_position_variance"
            ].values

            compensatory_post_var = compensatory_saccades[
                "post_saccade_position_variance"
            ].values

            post_var_stat, post_var_p = stats.mannwhitneyu(
                orienting_post_var, compensatory_post_var, alternative="two-sided"
            )

            print("\n  Post-saccade position variance (px²):")

            print(
                f"    Orienting: {orienting_post_var.mean():.2f} ± {orienting_post_var.std():.2f} (median: {np.median(orienting_post_var):.2f})"
            )

            print(
                f"    Compensatory: {compensatory_post_var.mean():.2f} ± {compensatory_post_var.std():.2f} (median: {np.median(compensatory_post_var):.2f})"
            )

            print(f"    Mann-Whitney U test: U={post_var_stat:.1f}, p={post_var_p:.4f}")

            # Bout size for compensatory saccades

            if len(compensatory_saccades) > 0:
                bout_sizes = compensatory_saccades["bout_size"].values
                print("\n  Bout size (compensatory saccades only):")
                print(
                    f"    Mean: {bout_sizes.mean():.2f} ± {bout_sizes.std():.2f} saccades"
                )
                print(
                    f"    Range: {bout_sizes.min():.0f} - {bout_sizes.max():.0f} saccades"
                )
                print(f"    Median: {np.median(bout_sizes):.0f} saccades")

            # Classification confidence comparison
            if "classification_confidence" in all_saccades_df.columns:
                orienting_conf = orienting_saccades["classification_confidence"].values

                compensatory_conf = compensatory_saccades[
                    "classification_confidence"
                ].values

                conf_stat, conf_p = stats.mannwhitneyu(
                    orienting_conf, compensatory_conf, alternative="two-sided"
                )

                print("\n  Classification Confidence:")

                print(
                    f"    Orienting: {orienting_conf.mean():.3f} ± {orienting_conf.std():.3f} (median: {np.median(orienting_conf):.3f})"
                )

                print(
                    f"    Compensatory: {compensatory_conf.mean():.3f} ± {compensatory_conf.std():.3f} (median: {np.median(compensatory_conf):.3f})"
                )

                print(f"    Mann-Whitney U test: U={conf_stat:.1f}, p={conf_p:.4f}")

        else:
            print(
                "  ⚠️ Cannot perform statistical comparisons - need both types present"
            )

        # Visualization

        # Create visualization figure

        fig_class = make_subplots(
            rows=2,
            cols=3,
            subplot_titles=(
                "Amplitude Distribution",
                "Duration Distribution",
                "Pre-saccade Velocity Distribution",
                "Pre-saccade Position Drift",
                "Post-saccade Position Variance",
                "Bout Size Distribution (Compensatory)",
            ),
            vertical_spacing=0.12,
            horizontal_spacing=0.1,
        )

        # Row 1, Col 1: Amplitude distributions

        if len(orienting_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=orienting_saccades["amplitude"],
                    nbinsx=30,
                    name="Orienting",
                    marker_color="blue",
                    opacity=0.6,
                    histnorm="probability",
                ),
                row=1,
                col=1,
            )

        if len(compensatory_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=compensatory_saccades["amplitude"],
                    nbinsx=30,
                    name="Compensatory",
                    marker_color="orange",
                    opacity=0.6,
                    histnorm="probability",
                ),
                row=1,
                col=1,
            )

        # Row 1, Col 2: Duration distributions

        if len(orienting_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=orienting_saccades["duration"],
                    nbinsx=30,
                    name="Orienting",
                    marker_color="blue",
                    opacity=0.6,
                    showlegend=False,
                    histnorm="probability",
                ),
                row=1,
                col=2,
            )
        if len(compensatory_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=compensatory_saccades["duration"],
                    nbinsx=30,
                    name="Compensatory",
                    marker_color="orange",
                    opacity=0.6,
                    showlegend=False,
                    histnorm="probability",
                ),
                row=1,
                col=2,
            )

        # Row 1, Col 3: Pre-saccade velocity distributions

        if len(orienting_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=orienting_saccades["pre_saccade_mean_velocity"],
                    nbinsx=30,
                    name="Orienting",
                    marker_color="blue",
                    opacity=0.6,
                    showlegend=False,
                    histnorm="probability",
                ),
                row=1,
                col=3,
            )

        if len(compensatory_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=compensatory_saccades["pre_saccade_mean_velocity"],
                    nbinsx=30,
                    name="Compensatory",
                    marker_color="orange",
                    opacity=0.6,
                    showlegend=False,
                    histnorm="probability",
                ),
                row=1,
                col=3,
            )

        # Row 2, Col 1: Pre-saccade drift distributions

        if len(orienting_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=orienting_saccades["pre_saccade_position_drift"],
                    nbinsx=30,
                    name="Orienting",
                    marker_color="blue",
                    opacity=0.6,
                    showlegend=False,
                    histnorm="probability",
                ),
                row=2,
                col=1,
            )

        if len(compensatory_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=compensatory_saccades["pre_saccade_position_drift"],
                    nbinsx=30,
                    name="Compensatory",
                    marker_color="orange",
                    opacity=0.6,
                    showlegend=False,
                    histnorm="probability",
                ),
                row=2,
                col=1,
            )

        # Row 2, Col 2: Post-saccade variance distributions

        if len(orienting_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=orienting_saccades["post_saccade_position_variance"],
                    nbinsx=30,
                    name="Orienting",
                    marker_color="blue",
                    opacity=0.6,
                    showlegend=False,
                    histnorm="probability",
                ),
                row=2,
                col=2,
            )

        if len(compensatory_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=compensatory_saccades["post_saccade_position_variance"],
                    nbinsx=30,
                    name="Compensatory",
                    marker_color="orange",
                    opacity=0.6,
                    showlegend=False,
                    histnorm="probability",
                ),
                row=2,
                col=2,
            )

        # Row 2, Col 3: Bout size distribution (compensatory only)

        if len(compensatory_saccades) > 0:
            fig_class.add_trace(
                go.Histogram(
                    x=compensatory_saccades["bout_size"],
                    nbinsx=20,
                    name="Compensatory Bout Size",
                    marker_color="orange",
                    opacity=0.6,
                    showlegend=False,
                    histnorm="probability",
                ),
                row=2,
                col=3,
            )

        else:
            # Add empty trace to maintain layout
            fig_class.add_trace(
                go.Histogram(x=[], name="No compensatory saccades"), row=2, col=3
            )

        # Update layout

        fig_class.update_layout(
            title_text=f"Saccade Classification Analysis: Orienting vs Compensatory ({get_eye_label(video_key)})",
            height=800,
            showlegend=True,
            legend=dict(x=0.02, y=0.98),
        )

        # Update axes labels
        fig_class.update_xaxes(title_text="Amplitude (px)", row=1, col=1)
        fig_class.update_xaxes(title_text="Duration (s)", row=1, col=2)
        fig_class.update_xaxes(title_text="Velocity (px/s)", row=1, col=3)
        fig_class.update_xaxes(title_text="Drift (px)", row=2, col=1)
        fig_class.update_xaxes(title_text="Variance (px²)", row=2, col=2)
        fig_class.update_xaxes(title_text="Bout Size (saccades)", row=2, col=3)

        fig_class.update_yaxes(title_text="Probability", row=1, col=1)
        fig_class.update_yaxes(title_text="Probability", row=1, col=2)
        fig_class.update_yaxes(title_text="Probability", row=1, col=3)
        fig_class.update_yaxes(title_text="Probability", row=2, col=1)
        fig_class.update_yaxes(title_text="Probability", row=2, col=2)
        fig_class.update_yaxes(title_text="Probability", row=2, col=3)

        fig_class.show()

        # Confidence distribution visualization

        if "classification_confidence" in all_saccades_df.columns:
            fig_conf = make_subplots(
                rows=1,
                cols=2,
                subplot_titles=(
                    "Classification Confidence Distribution",
                    "Confidence by Saccade Type",
                ),
                horizontal_spacing=0.15,
            )

        # Overall confidence distribution

        fig_conf.add_trace(
            go.Histogram(
                x=all_saccades_df["classification_confidence"],
                nbinsx=30,
                name="All Saccades",
                marker_color="gray",
                opacity=0.7,
                histnorm="probability",
            ),
            row=1,
            col=1,
        )

        # Add vertical lines for confidence thresholds
        fig_conf.add_vline(
            x=0.7,
            line_dash="dash",
            line_color="green",
            annotation_text="High (≥0.7)",
            row=1,
            col=1,
        )
        fig_conf.add_vline(
            x=0.4,
            line_dash="dash",
            line_color="orange",
            annotation_text="Medium (0.4-0.7)",
            row=1,
            col=1,
        )

        # Confidence by type
        if len(orienting_saccades) > 0:
            fig_conf.add_trace(
                go.Histogram(
                    x=orienting_saccades["classification_confidence"],
                    nbinsx=30,
                    name="Orienting",
                    marker_color="blue",
                    opacity=0.6,
                    histnorm="probability",
                ),
                row=1,
                col=2,
            )
        if len(compensatory_saccades) > 0:
            fig_conf.add_trace(
                go.Histogram(
                    x=compensatory_saccades["classification_confidence"],
                    nbinsx=30,
                    name="Compensatory",
                    marker_color="orange",
                    opacity=0.6,
                    histnorm="probability",
                ),
                row=1,
                col=2,
            )

        fig_conf.update_layout(
            title_text=f"Classification Confidence Analysis ({get_eye_label(video_key)})",
            height=400,
            showlegend=True,
            legend=dict(x=0.02, y=0.98),
        )

        fig_conf.update_xaxes(title_text="Confidence Score", row=1, col=1)

        fig_conf.update_xaxes(title_text="Confidence Score", row=1, col=2)

        fig_conf.update_yaxes(title_text="Probability", row=1, col=1)

        fig_conf.update_yaxes(title_text="Probability", row=1, col=2)

        fig_conf.show()

        # Time series visualization with classification

        fig_ts = make_subplots(
            rows=2,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.1,
            subplot_titles=(
                "X Position (px)",
                "Velocity (px/s) with Classified Saccades",
            ),
        )

        # Add position trace

        fig_ts.add_trace(
            go.Scatter(
                x=res["df"]["Seconds"],
                y=res["df"]["X_smooth"],
                mode="lines",
                name="Smoothed X",
                line=dict(color="blue", width=2),
            ),
            row=1,
            col=1,
        )

        # Add velocity trace

        fig_ts.add_trace(
            go.Scatter(
                x=res["df"]["Seconds"],
                y=res["df"]["vel_x_smooth"],
                mode="lines",
                name="Smoothed Velocity",
                line=dict(color="red", width=2),
            ),
            row=2,
            col=1,
        )

        # Add adaptive threshold lines
        fig_ts.add_hline(
            y=res["vel_thresh"],
            line_dash="dash",
            line_color="green",
            opacity=0.5,
            annotation_text=f"Adaptive threshold (±{res['vel_thresh']:.0f} px/s)",
            row=2,
            col=1,
        )
        fig_ts.add_hline(
            y=-res["vel_thresh"],
            line_dash="dash",
            line_color="green",
            opacity=0.5,
            row=2,
            col=1,
        )

        # Calculate position y-axis range for vertical lines
        pos_max = res["df"]["X_smooth"].max()
        pos_min = res["df"]["X_smooth"].min()
        pos_range = pos_max - pos_min
        # Add small padding to ensure lines span full visible range
        pos_padding = pos_range * 0.05
        pos_y_min = pos_min - pos_padding
        pos_y_max = pos_max + pos_padding

        # Plot orienting saccades (blue) as rectangles on position trace
        orienting_in_df = all_saccades_df[
            all_saccades_df["saccade_type"] == "orienting"
        ]
        if len(orienting_in_df) > 0:
            for idx, row in orienting_in_df.iterrows():
                start_time = row["start_time"]

                end_time = row["end_time"]

                # Use rectangle with thin border to show duration span more
                # efficiently

                # Opacity must be set via rgba color, not in line dict

                fig_ts.add_shape(
                    type="rect",
                    x0=start_time,
                    y0=pos_y_min,
                    x1=end_time,
                    y1=pos_y_max,
                    fillcolor="rgba(0,0,255,0.1)",  # Light blue fill with opacity
                    # Blue border with opacity via rgba
                    line=dict(color="rgba(0,0,255,0.3)", width=2),
                    row=1,
                    col=1,
                )

        # Legend entry

        fig_ts.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="lines",
                name="Orienting Saccades",
                line=dict(
                    color="rgba(0,0,255,0.3)", width=3
                ),  # Blue with opacity via rgba
            ),
            row=1,
            col=1,
        )

        # Plot compensatory saccades (orange) as rectangles on position trace

        compensatory_in_df = all_saccades_df[
            all_saccades_df["saccade_type"] == "compensatory"
        ]

        if len(compensatory_in_df) > 0:
            for idx, row in compensatory_in_df.iterrows():
                start_time = row["start_time"]
                end_time = row["end_time"]

                # Use rectangle with thin border to show duration span more
                # efficiently

                # Opacity must be set via rgba color, not in line dict

                fig_ts.add_shape(
                    type="rect",
                    x0=start_time,
                    y0=pos_y_min,
                    x1=end_time,
                    y1=pos_y_max,
                    # Light orange fill with opacity
                    fillcolor="rgba(255,165,0,0.1)",
                    # Orange border with opacity via rgba
                    line=dict(color="rgba(255,165,0,0.3)", width=2),
                    row=1,
                    col=1,
                )

        # Legend entry

        fig_ts.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="lines",
                name="Compensatory Saccades",
                line=dict(
                    color="rgba(255,165,0,0.3)", width=3
                ),  # Orange with opacity via rgba
            ),
            row=1,
            col=1,
        )

        # Update layout
        fig_ts.update_layout(
            title=f"Time Series with Saccade Classification ({get_eye_label(video_key)})<br><sub>Blue: Orienting, Orange: Compensatory</sub>",
            height=600,
            showlegend=True,
            legend=dict(x=0.01, y=0.99),
        )

        # Update axes
        fig_ts.update_xaxes(title_text="Time (s)", row=2, col=1)
        fig_ts.update_yaxes(title_text="X Position (px)", row=1, col=1)
        fig_ts.update_yaxes(title_text="Velocity (px/s)", row=2, col=1)

        fig_ts.show()

In [ ]:
# Gracefully stop execution by informing the user and returning from the cell
sys.exit("⚠️ Notebook execution intentionally stopped here. Remove or move this cell to continue running subsequent code.")

In [ ]:
# ML Feature Extraction and Visualization
# Extract features for ML classification and visualize distributions
##########################################################################


# Extract features for VideoData1 (if available)
features_v1 = None
if VideoData1_Has_Sleap and "VideoData1" in saccade_results:
    print("=" * 80)
    print("Extracting ML features for VideoData1...")
    print("=" * 80)
    # Use the processed df from saccade_results (has X_smooth, vel_x_smooth
    # columns)
    df1_processed = saccade_results["VideoData1"]["df"]
    features_v1 = extract_ml_features(
        saccade_results=saccade_results["VideoData1"],
        df=df1_processed,  # Use processed df with X_smooth, vel_x_smooth
        fps=FPS_1,
        data_path=data_path,
        verbose=True,
    )
    if len(features_v1) > 0:
        features_v1["eye"] = "Left" if video1_eye == "L" else "Right"
        print(
            f"✅ Extracted {len(features_v1)} saccades with {len(features_v1.columns)} features"
        )

# Extract features for VideoData2 (if available)
features_v2 = None
if VideoData2_Has_Sleap and "VideoData2" in saccade_results:
    print("\n" + "=" * 80)
    print("Extracting ML features for VideoData2...")
    print("=" * 80)
    # Use the processed df from saccade_results (has X_smooth, vel_x_smooth
    # columns)
    df2_processed = saccade_results["VideoData2"]["df"]
    features_v2 = extract_ml_features(
        saccade_results=saccade_results["VideoData2"],
        df=df2_processed,  # Use processed df with X_smooth, vel_x_smooth
        fps=FPS_2,
        data_path=data_path,
        verbose=True,
    )
    if len(features_v2) > 0:
        features_v2["eye"] = "Right" if video1_eye == "L" else "Left"
        print(
            f"✅ Extracted {len(features_v2)} saccades with {len(features_v2.columns)} features"
        )

# Combine features from both eyes
if features_v1 is not None and features_v2 is not None:
    features_combined = pd.concat([features_v1, features_v2], ignore_index=True)
    print(
        f"\n✅ Combined features: {len(features_combined)} total saccades ({len(features_v1)} left, {len(features_v2)} right)"
    )
elif features_v1 is not None:
    features_combined = features_v1.copy()
    print(f"\n✅ Using VideoData1 only: {len(features_combined)} saccades")
elif features_v2 is not None:
    features_combined = features_v2.copy()
    print(f"\n✅ Using VideoData2 only: {len(features_combined)} saccades")
else:
    print("\n⚠️ No features extracted - check if saccades were detected")
    features_combined = None

In [ ]:
# Visualize Feature Distributions: Panel 1 (Violin Plots by Category) + Panel 2 (Key Features by Class)
##########################################################################

if features_combined is not None and len(features_combined) > 0:
    # Define feature categories
    feature_categories = {
        "Category A: Basic Properties": [
            "amplitude",
            "duration",
            "peak_velocity",
            "direction",
            "start_time",
            "end_time",
            "time",
        ],
        "Category B: Pre-Saccade": [
            "pre_saccade_mean_velocity",
            "pre_saccade_position_drift",
            "pre_saccade_position_variance",
            "pre_saccade_drift_rate",
            "pre_saccade_window_duration",
        ],
        "Category C: Post-Saccade": [
            "post_saccade_position_variance",
            "post_saccade_position_change",
            "post_saccade_position_change_pct",
            "post_saccade_mean_velocity",
            "post_saccade_drift_rate",
            "post_saccade_window_duration",
        ],
        "Category D: Temporal Context": [
            "time_since_previous_saccade",
            "time_until_next_saccade",
            "bout_size",
            "position_in_bout",
            "is_first_in_bout",
            "is_last_in_bout",
            "is_isolated",
            "bout_duration",
            "inter_saccade_interval_mean",
            "inter_saccade_interval_std",
        ],
        "Category G: Amplitude/Direction Consistency": [
            "amplitude_relative_to_bout_mean",
            "amplitude_consistency_in_bout",
            "direction_relative_to_previous",
        ],
        "Category H: Rule-Based Classification": [
            "rule_based_class",
            "rule_based_confidence",
        ],
    }

    # Use the visualization function
    visualize_ml_features(
        features_combined=features_combined,
        feature_categories=feature_categories,
        video_labels=VIDEO_LABELS,
        show_plots=True,
        verbose=True,
    )
else:
    print("⚠️ No features available for visualization. Run feature extraction first.")

In [ ]:
# LAUNCH GUI ANNOTATION TOOL
##########################################################################
# Use this cell to launch the GUI for manual annotation of saccades
# The GUI starts with rule-based classifications and allows you to correct
# them to 4 classes

# Get experiment ID
experiment_id = extract_experiment_id(data_path)
print(f"Experiment ID: {experiment_id}")

# Launch GUI with BOTH eyes combined (VideoData1 and VideoData2)
# The GUI will automatically combine saccades from both eyes and display
# them together

# Get features for BOTH eyes (since we're combining both eyes' saccades)
# features_combined already contains features from both VideoData1 and
# VideoData2
video_features = (
    features_combined
    if features_combined is not None and len(features_combined) > 0
    else None
)

# Set annotations file path (save in data_path parent directory)
annotations_file = data_path.parent / "saccade_annotations_master.csv"
# Or use a project-wide location:
# annotations_file = Path('/Users/rancze/Documents/Data/vestVR/saccade_annotations_master.csv')

# Count saccades from both eyes for display
total_saccades = 0
eye_breakdown = {}
if isinstance(saccade_results, dict):
    for key in ["VideoData1", "VideoData2"]:
        if key in saccade_results:
            eye_data = saccade_results[key]
            if "all_saccades_df" in eye_data:
                count = len(eye_data["all_saccades_df"])
                total_saccades += count
                eye_label = VIDEO_LABELS.get(key, key)
                eye_breakdown[eye_label] = count

print(f"\n{'=' * 80}")
print("Launching GUI Annotation Tool (BOTH EYES)")
print(f"{'=' * 80}")
print(f"Experiment ID: {experiment_id}")
print(f"Eyes: {', '.join(eye_breakdown.keys()) if eye_breakdown else 'Unknown'}")
print(f"Total saccades: {total_saccades}")
if eye_breakdown:
    for eye, count in eye_breakdown.items():
        print(f"  - {eye}: {count} saccades")
print(f"Features: {len(video_features) if video_features is not None else 0}")
print(f"Annotations file: {annotations_file}")
print(f"{'=' * 80}\n")
print("ℹ️ Instructions:")
print(
    "  - Use keyboard shortcuts: 1=Compensatory, 2=Orienting, 3=Saccade-and-Fixate, 4=Non-Saccade"
)
print("  - Navigation: . = Next, , = Previous, S = Save")
print("  - Click on saccades in the table to select them")
print("  - The table shows saccades from both eyes (L and R) combined")
print("  - Close the GUI window when done annotating")
print(f"{'=' * 80}\n")

# Launch GUI (this will block until GUI is closed)
# Pass the FULL saccade_results dict so both eyes are combined
launch_annotation_gui(
    # Pass full dict with VideoData1 and VideoData2 keys
    saccade_results=saccade_results,
    features_df=video_features,  # Pass all features (both eyes)
    experiment_id=experiment_id,
    annotations_file_path=annotations_file,
)

# After GUI closes, show statistics
print(f"\n{'=' * 80}")
print("Annotation session complete!")
print(f"{'=' * 80}")

# Load and display annotations
annotations = load_annotations(annotations_file, experiment_id=experiment_id)
if len(annotations) > 0:
    print(f"\n✅ Annotated {len(annotations)} saccades for this experiment")
    print_annotation_stats(annotations_file, experiment_id=experiment_id)
else:
    print("\n⚠️ No annotations saved for this experiment")